# Imports

In [1]:
from _spo_utils import camel_to_snake, import_json

import pandas as pd 
import json

# Constants

In [ ]:
# PATH_SPOTIFY = '../../data/2_processed/final_df.csv'
PATH_SPOTIFY = '../../data/2_processed/consolidated.csv'
PATH_MARQUEE = '../../data/1_raw/Marquee.json'
PATH_HISTORY = '../../data/1_raw/Streaming_History_Audio.json'
PATH_LIBRARY = '../../data/1_raw/YourLibrary.json'
PATH_PLAYLIST = '../../data/1_raw/Playlist1.json'

<h1>JSON File Convensions & Imports</h1>

<h3 style='color:gray;'>Imports Marquee JSON file (1/4)</h3>

In [3]:
marquee = import_json(PATH_MARQUEE)
marquee.head()

,artist_name,segment
0,Cobrah,Previously Active Listeners
1,BLACKPINK,Previously Active Listeners
2,Martin Garrix,Previously Active Listeners
3,Confetti,Previously Active Listeners
4,Demi Lovato,Previously Active Listeners


<h3 style='color:gray;'>Imports Streaming History JSON file, using record_path (2/4)</h3>

In [4]:
# History — simple
history = import_json(PATH_HISTORY)
history.head()

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2020-10-20T19:36:53Z,"Android OS 10 API 29 (samsung, SM-A307GT)",23600,BR,177.58.181.120,Pretty Savage,BLACKPINK,THE ALBUM,spotify:track:1XnpzbOGptRwfJhZgLbmSr,None,...,None,None,None,clickrow,backbtn,False,False,False,NaN,False
1,2020-10-20T19:36:55Z,"Android OS 10 API 29 (samsung, SM-A307GT)",1052,BR,177.58.181.120,How You Like That,BLACKPINK,THE ALBUM,spotify:track:4SFknyjLcyTLJFPKD2m96o,None,...,None,None,None,backbtn,backbtn,False,False,False,NaN,False
2,2020-10-20T19:36:56Z,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Ice Cream (with Selena Gomez),BLACKPINK,THE ALBUM,spotify:track:4JUPEh2DVSXFGExu4Uxevz,None,...,None,None,None,backbtn,backbtn,False,False,False,NaN,False
3,2020-10-20T19:36:56Z,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Bet You Wanna (feat. Cardi B),BLACKPINK,THE ALBUM,spotify:track:7iAgNZdotu40NwtoIWJHFe,None,...,None,None,None,backbtn,backbtn,False,False,False,NaN,False
4,2020-10-20T19:36:59Z,"Android OS 10 API 29 (samsung, SM-A307GT)",1824,BR,177.58.181.120,Lovesick Girls,BLACKPINK,THE ALBUM,spotify:track:4Ws314Ylb27BVsvlZOy30C,None,...,None,None,None,backbtn,fwdbtn,False,False,False,NaN,False


<h3 style='color:gray;'>Imports Library JSON file, using record_path (3/4)</h3>

In [5]:
library = import_json(PATH_LIBRARY, record_path=['tracks'])
library.head()

,artist,album,track,uri
0,Set It Off,Wolf In Sheep's Clothing [REBORN],Wolf In Sheep's Clothing [REBORN],spotify:track:1tpidJ4FBn9TwshePh1bc3
1,Skillet,Awake,Monster,spotify:track:2UREu1Y8CO4jXkbvqAtP7g
2,Thousand Foot Krutch,The End Is Where We Begin,Courtesy Call,spotify:track:0AOmbw8AwDnwXhHC3OhdVB


<h3 style='color:gray;'>Imports Playlist JSON file, using extra method (4/4)</h3>

In [6]:
def transform_playlist(data):
    all_playlists = []
    for i in data['playlists']:
        i_playlist = pd.json_normalize(i['items'], sep='_')
        i_playlist['playlists_lastModifiedDate'] = i['lastModifiedDate']
        i_playlist['playlists_name'] = i['name']
        all_playlists.append(i_playlist)
    return pd.concat(all_playlists)

playlist = import_json(PATH_PLAYLIST, transform=transform_playlist)
playlist.head()

,episode,audiobook,local_track,added_date,track_track_name,track_artist_name,track_album_name,track_track_uri,track,episode_episode_name,episode_show_name,episode_episode_uri,playlists_last_modified_date,playlists_name
0,NaN,None,None,2024-10-22,Angry Too,Lola Blanc,Angry Too,spotify:track:51jK7uI2QR8JCP4G9DnbQS,NaN,NaN,NaN,NaN,2026-01-18,<3
1,NaN,None,None,2024-10-22,Walls Could Talk,Halsey,hopeless fountain kingdom,spotify:track:1Ao5nOuAUm42ob1UVvOkgW,NaN,NaN,NaN,NaN,2026-01-18,<3
2,NaN,None,None,2024-10-22,Freak (feat. REI AMI),Sub Urban,Thrill Seeker,spotify:track:5jkbw3FDS3bSSb4oLKYbjf,NaN,NaN,NaN,NaN,2026-01-18,<3
3,NaN,None,None,2024-10-22,Shameless,Camila Cabello,Romance,spotify:track:2ogKhhoMClkFXek7ZgxAhN,NaN,NaN,NaN,NaN,2026-01-18,<3
4,NaN,None,None,2024-10-22,Looking at Me,Sabrina Carpenter,Singular Act II,spotify:track:59tskctgqUmjCWAwhzYAFm,NaN,NaN,NaN,NaN,2026-01-18,<3


# Data Consolidation

In [7]:
# 1. Merge history and marquee
final_df = pd.merge(
    history, 
    marquee, 
    how='left', 
    left_on='master_metadata_album_artist_name', 
    right_on='artist_name'
).drop(columns=['master_metadata_album_artist_name'])

# 2. Drop nulls EARLY (before applying string operations to save compute time)
final_df = final_df.dropna(subset=['spotify_track_uri'])

# 3. Extract track_id (str.split is generally more idiomatic and readable than rpartition)
final_df['track_id'] = final_df['spotify_track_uri'].str.split(':').str[-1]

# 4. Save to CSV
final_df.to_csv(PATH_SPOTIFY, index=False)

final_df.head()

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,...,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode,artist_name,segment,track_id
0,2020-10-20T19:36:53Z,"Android OS 10 API 29 (samsung, SM-A307GT)",23600,BR,177.58.181.120,Pretty Savage,THE ALBUM,spotify:track:1XnpzbOGptRwfJhZgLbmSr,None,None,...,clickrow,backbtn,False,False,False,NaN,False,BLACKPINK,Previously Active Listeners,1XnpzbOGptRwfJhZgLbmSr
1,2020-10-20T19:36:55Z,"Android OS 10 API 29 (samsung, SM-A307GT)",1052,BR,177.58.181.120,How You Like That,THE ALBUM,spotify:track:4SFknyjLcyTLJFPKD2m96o,None,None,...,backbtn,backbtn,False,False,False,NaN,False,BLACKPINK,Previously Active Listeners,4SFknyjLcyTLJFPKD2m96o
2,2020-10-20T19:36:56Z,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Ice Cream (with Selena Gomez),THE ALBUM,spotify:track:4JUPEh2DVSXFGExu4Uxevz,None,None,...,backbtn,backbtn,False,False,False,NaN,False,BLACKPINK,Previously Active Listeners,4JUPEh2DVSXFGExu4Uxevz
3,2020-10-20T19:36:56Z,"Android OS 10 API 29 (samsung, SM-A307GT)",0,BR,177.58.181.120,Bet You Wanna (feat. Cardi B),THE ALBUM,spotify:track:7iAgNZdotu40NwtoIWJHFe,None,None,...,backbtn,backbtn,False,False,False,NaN,False,BLACKPINK,Previously Active Listeners,7iAgNZdotu40NwtoIWJHFe
4,2020-10-20T19:36:59Z,"Android OS 10 API 29 (samsung, SM-A307GT)",1824,BR,177.58.181.120,Lovesick Girls,THE ALBUM,spotify:track:4Ws314Ylb27BVsvlZOy30C,None,None,...,backbtn,fwdbtn,False,False,False,NaN,False,BLACKPINK,Previously Active Listeners,4Ws314Ylb27BVsvlZOy30C
